# 🤖 Complete RAG System — Practical Learning Notebook

**Book used:** *As a Man Thinketh* by James Allen (1903) — 100% Free & Public Domain  
**LLM:** Ollama running locally (no API key needed, no internet required)  
**Framework:** LangChain 1.x  
**Vector Store:** ChromaDB  

---

## 📋 What You Will Build Step by Step:
```
📄 Text Document
       ↓
✂️  Chunking (split into pieces)
       ↓
🔢 Embeddings (convert text → numbers/vectors)
       ↓
🗄️  Vector Store (ChromaDB — store the vectors)
       ↓
🔍 Retrieval (find relevant chunks for a question)
       ↓
🦙 LLM via Ollama (generate answer using retrieved context)
       ↓
💬 Chat Interface (terminal-based Q&A loop)
```

---
⚠️ **Pre-requisite:** Make sure Ollama is installed and these models are pulled:
```
ollama pull llama3.2
ollama pull nomic-embed-text
```


---
## 🔧 CELL 1 — Install All Dependencies
Run this **once**. After installation, you can skip it on subsequent runs.

In [1]:
import subprocess, sys

packages = [
    "langchain>=0.2.0",
    "langchain-core",
    "langchain-ollama",
    "langchain-text-splitters",
    "langchain-chroma",
    "chromadb>=0.5.0",
    "pypdf>=4.0.0",
    "langchain-community",
    "requests",
    "tqdm",
]

for pkg in packages:
    print(f"📦 Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed successfully!")

📦 Installing langchain>=0.2.0...
📦 Installing langchain-core...
📦 Installing langchain-ollama...
📦 Installing langchain-text-splitters...
📦 Installing langchain-chroma...
📦 Installing chromadb>=0.5.0...
📦 Installing pypdf>=4.0.0...
📦 Installing langchain-community...
📦 Installing requests...
📦 Installing tqdm...

✅ All packages installed successfully!


---
## 📥 CELL 2 — Download the Book (Public Domain)
*As a Man Thinketh* by James Allen — published 1903, completely free & legal to use.

In [2]:
import requests
import os

BOOK_URL  = "https://www.gutenberg.org/cache/epub/4507/pg4507.txt"
DATA_DIR  = "../data"
BOOK_PATH = os.path.join(DATA_DIR, "as_a_man_thinketh.txt")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(BOOK_PATH):
    print("📥 Downloading book from Project Gutenberg...")
    response = requests.get(BOOK_URL, timeout=30)
    response.raise_for_status()
    with open(BOOK_PATH, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✅ Book saved to: {BOOK_PATH}")
else:
    print(f"✅ Book already exists at: {BOOK_PATH}")

size_kb = os.path.getsize(BOOK_PATH) / 1024
print(f"📄 File size: {size_kb:.1f} KB")

with open(BOOK_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"📝 Total characters: {len(raw_text):,}")
print("\n--- BOOK PREVIEW (first 400 characters) ---")
print(raw_text[:400])

✅ Book already exists at: ../data\as_a_man_thinketh.txt
📄 File size: 65.4 KB
📝 Total characters: 65,526

--- BOOK PREVIEW (first 400 characters) ---
The Project Gutenberg eBook of As a man thinketh

    

This eBook is for the use of anyone anywhere in the United States and

most other parts of the world at no cost and with almost no restrictions

whatsoever. You may copy it, give it away or re-use it under the terms

of the Project Gutenberg License included with this eBook or online

at www.gutenberg.org. If you are not located in the United


---
## 📂 CELL 3 — Load the Document with LangChain
LangChain wraps files into `Document` objects — each has `page_content` (text) and `metadata` (source info).

In [3]:
# ✅ Flexible loader: Loads either PDF or TXT seamlessly using LangChain loaders
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader

PDF_PATH = os.path.join(DATA_DIR, 'as_a_man_thinketh.pdf')

if os.path.exists(PDF_PATH):
    print(f'📂 Loading PDF document from: {PDF_PATH}')
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()
else:
    print(f'📂 Loading TXT document from: {BOOK_PATH}')
    loader = TextLoader(BOOK_PATH, encoding='utf-8')
    documents = loader.load()

total_chars = sum(len(d.page_content) for d in documents)
print(f'✅ Loaded {len(documents)} page/document(s)')
print(f'📝 Total characters: {total_chars:,}')
print(f'🏷️  Metadata (Doc 0): {documents[0].metadata}')
print('\n--- First 300 characters of Doc 0 ---')
print(documents[0].page_content[:300].strip())


C:\Users\SergioMarquina\AppData\Local\Temp\ipykernel_9648\3457174399.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


📂 Loading PDF document from: ../data\as_a_man_thinketh.pdf
✅ Loaded 32 page/document(s)
📝 Total characters: 47,245
🏷️  Metadata (Doc 0): {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-03T21:21:00+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-03T21:21:00+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data\\as_a_man_thinketh.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}

--- First 300 characters of Doc 0 ---
AS A MAN THINKETH
 A Classic Philosophical & Mindset Essay
 BY JAMES ALLEN
 Published 1903  100% Free & Open Public Domain Document
 Complete Text for RAG Document Ingestion & Vector Search Experiments


---
## ✂️ CELL 4 — CHUNKING: Splitting Text into Pieces

### Why chunk?
- LLMs have a **context window limit** — can't process an entire book at once
- Embedding large text loses specificity — small chunks embed more accurately
- We retrieve only **relevant** chunks, not the whole book

```
Full Text:  [================ entire book ================]
Chunk 1:    [========]         ← chunk_size characters
Chunk 2:       [========]      ← overlaps with chunk 1 by chunk_overlap chars
Chunk 3:          [========]
```

In [4]:
# ✅ RecursiveCharacterTextSplitter from langchain_text_splitters (LangChain 1.x)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # Each chunk: at most 800 characters
    chunk_overlap=150,     # 150 characters shared between adjacent chunks
    length_function=len,
    separators=['\n\n', '\n', ' ', '']  # Try these split points in order
)

chunks = text_splitter.split_documents(documents)

total_doc_chars = sum(len(d.page_content) for d in documents)
print(f'📊 Chunking Results:')
print(f'   Original : {len(documents)} page/document(s), {total_doc_chars:,} characters')
print(f'   After    : {len(chunks)} chunks')
print(f'   Average  : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars/chunk')
print()

for i, chunk in enumerate(chunks[5:8], start=5):
    print(f'--- CHUNK #{i} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print()


📊 Chunking Results:
   Original : 32 page/document(s), 47,245 characters
   After    : 84 chunks
   Average  : 638 chars/chunk

--- CHUNK #5 (757 chars) ---
are deliberately executed.
Act is the blossom of thought, and joy and suffering are its fruits;
thus does a man garner in the sweet and bitter fruitage of his own
husbandry.
"Thought in the mind hath made us, What we are
By thought was wrought and built. If a man's mind
Hath evil thoughts, pain comes on him as comes
The wheel the ox behind....
..If one endure
In purity of thought, joy follows him
As his own shadow--sure."
Man is a growth by law, and not a creation by artifice, and cause
and effect is as absolute and undeviating in the hidden realm of
thought as in the world of visible and material things. A noble and
Godlike character is not a thing of favour or chance, but is the
natural result of continued effort in right thinking, the effect of

--- CHUNK #6 (796 chars) ---
Godlike character is not a thing of favour or chance, b

---
## 🔢 CELL 5 — EMBEDDINGS: Converting Text to Vectors

An **embedding** is a list of numbers that captures the *meaning* of text.  
Similar meaning → similar numbers → can be found by similarity search.

```
"Man is shaped by his thoughts"  →  [0.12, -0.34, 0.89, ...]  (768 numbers)
"Character is built by thinking" →  [0.11, -0.30, 0.85, ...]  ← similar!
"The cat sat on the mat"         →  [-0.88, 0.45, -0.12, ...] ← very different
```

> ⚠️ First run in terminal: `ollama pull nomic-embed-text`

In [5]:
# ✅ langchain_ollama is the correct package for Ollama in LangChain 1.x
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

test_sentence = "A man's mind is like a garden that must be tended."
print(f"📝 Test sentence: '{test_sentence}'")
print("🔄 Generating embedding (calls Ollama locally)...")

vector = embedding_model.embed_query(test_sentence)

print(f"\n✅ Embedding generated!")
print(f"📏 Dimensions : {len(vector)}")
print(f"🔢 First 8 values: {[round(v, 4) for v in vector[:8]]}")
print()
print("💡 Similar texts → similar vectors → similarity search finds them!")

📝 Test sentence: 'A man's mind is like a garden that must be tended.'
🔄 Generating embedding (calls Ollama locally)...

✅ Embedding generated!
📏 Dimensions : 768
🔢 First 8 values: [0.0444, 0.0603, -0.1528, -0.0613, 0.0412, 0.0145, 0.0228, -0.0047]

💡 Similar texts → similar vectors → similarity search finds them!


---
## 🗄️ CELL 6 — VECTOR STORE: Storing Embeddings in ChromaDB

ChromaDB stores `(text, vector)` pairs and supports **similarity search** — find the chunks whose vectors are closest to your question's vector.

```
ChromaDB:
┌──────────────────────────────────────────────────────────┐
│  id  │  Text (chunk)                  │  Vector           │
├──────────────────────────────────────────────────────────┤
│  001 │ "Man is made by his thoughts"  │  [0.12, -0.34...] │
│  002 │ "The body obeys the mind"      │  [0.88,  0.21...] │
│  ...                                                      │
└──────────────────────────────────────────────────────────┘
```
⏳ **First run takes 1–3 minutes** (embeds all chunks). Subsequent runs load from disk instantly.

In [6]:
# ✅ FIX: Use langchain_chroma (standalone package) instead of langchain_community
from langchain_chroma import Chroma

CHROMA_DIR = "../data/chroma_db"

print("🔄 Creating vector store...")
print(f"   📦 Embedding {len(chunks)} chunks using nomic-embed-text...")
print("   ⏳ This may take 1–3 minutes on first run...")
print()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name="book_knowledge"
)

print(f"✅ Vector store created and saved to disk!")
print(f"📂 Location       : {CHROMA_DIR}")
print(f"📊 Chunks stored  : {vectorstore._collection.count()}")

🔄 Creating vector store...
   📦 Embedding 84 chunks using nomic-embed-text...
   ⏳ This may take 1–3 minutes on first run...

✅ Vector store created and saved to disk!
📂 Location       : ../data/chroma_db
📊 Chunks stored  : 84


---
## 🔍 CELL 7 — RETRIEVAL: Finding the Most Relevant Chunks

```
User Question  →  embed question  →  find 3 closest vectors  →  return 3 chunks
```
The retrieved chunks become the **context** passed to the LLM.

In [7]:
# k=3 → return the 3 most relevant chunks per query
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

test_question = "What is the relationship between thought and character?"
print(f"❓ Test Question: {test_question}")
print()

retrieved_docs = retriever.invoke(test_question)

print(f"📚 Retrieved {len(retrieved_docs)} relevant chunks:")
print("=" * 60)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[CHUNK {i}]")
    print(doc.page_content)
    print("-" * 40)

print()
print("💡 These chunks are the CONTEXT sent to the LLM to generate the answer!")

❓ Test Question: What is the relationship between thought and character?

📚 Retrieved 3 relevant chunks:

[CHUNK 1]
ever-increasing accuracy, how the thought-forces and mind elements
operate in the shaping of his character, circumstances, and destiny.
Thought and character are one, and as character can only manifest
As a Man Thinketh — James Allen
Page 7 of 32
Public Domain (Project Gutenberg)
----------------------------------------

[CHUNK 2]
Godlike character is not a thing of favour or chance, but is the
natural result of continued effort in right thinking, the effect of
long-cherished association with Godlike thoughts. An ignoble and
bestial character, by the same process, is the result of the
continued harbouring of grovelling thoughts.
Man is made or unmade by himself; in the armoury of thought he
forges the weapons by which he destroys himself; he also fashions
the tools with which he builds for himself heavenly mansions of joy
and strength and peace. By the right choice and tr

---
## 🦙 CELL 8 — LLM Setup: Loading Ollama Locally

Ollama runs LLMs on your machine — no API key, no internet, no cost.

> ⚠️ First run in terminal: `ollama pull llama3.2`

In [8]:
# ✅ langchain_ollama is correct for LangChain 1.x
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.1,   # Low = more factual (good for RAG)
)

print("🔄 Testing Ollama connection...")
test_response = llm.invoke("Reply with exactly: 'Ollama is working!'")
print(f"✅ Ollama Response: {test_response}")
print()
print("🦙 LLM ready! Model: llama3.2")

🔄 Testing Ollama connection...
✅ Ollama Response: Ollama is working!

🦙 LLM ready! Model: llama3.2


---
## 🔗 CELL 9 — RAG CHAIN: Connecting Everything

```
Question → Retriever → [Context Chunks] → Prompt Template → LLM → Answer
```

The **Prompt Template** tells the LLM:
1. Here is context from the book
2. Here is the user's question  
3. Answer **only** from the context → prevents hallucination!

In [9]:
# ✅ FIX: All imports now from langchain_core (correct for LangChain 1.x)
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT_TEMPLATE = """\
You are a helpful assistant that answers questions based on the book "As a Man Thinketh" by James Allen.

Use ONLY the information from the context below to answer.
If the answer is not in the context, say: "The book does not cover this topic in the retrieved sections."
Keep answers clear and insightful.

CONTEXT FROM THE BOOK:
{context}

QUESTION: {question}

ANSWER:"""

prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

def format_docs(docs):
    """Join retrieved document chunks into one context string."""
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# LangChain Expression Language (LCEL) pipeline — pipe | connects each step
rag_chain = (
    {
        "context" : retriever | format_docs,  # Retrieve → format
        "question": RunnablePassthrough()      # Pass question unchanged
    }
    | prompt           # Fill the template
    | llm              # Call Ollama
    | StrOutputParser()  # Return clean string
)

print("✅ RAG Chain built!")
print()
print("Pipeline:")
print("  Question → Retriever (top 3 chunks)")
print("  Chunks   → format_docs (joined context)")
print("  Context + Question → PromptTemplate")
print("  Prompt   → OllamaLLM (llama3.2)")
print("  Output   → StrOutputParser (clean string)")

✅ RAG Chain built!

Pipeline:
  Question → Retriever (top 3 chunks)
  Chunks   → format_docs (joined context)
  Context + Question → PromptTemplate
  Prompt   → OllamaLLM (llama3.2)
  Output   → StrOutputParser (clean string)


---
## 💬 CELL 10 — Test with Sample Questions

In [10]:
test_questions = [
    "What does the book say about the relationship between thought and character?",
    "How does the mind affect our health and body?",
    "What is the importance of having a definite purpose in life?",
]

for i, question in enumerate(test_questions, 1):
    print(f"{'='*60}")
    print(f"❓ Q{i}: {question}")
    print(f"{'='*60}")
    print("🔄 Thinking...")
    answer = rag_chain.invoke(question)
    print(f"\n🤖 Answer:")
    print(answer)
    print()

❓ Q1: What does the book say about the relationship between thought and character?
🔄 Thinking...

🤖 Answer:
According to the book, "Thought and character are one, and as character can only manifest As a Man Thinketh." This means that thought and character are intimately connected, and the thoughts we think shape our character.

❓ Q2: How does the mind affect our health and body?
🔄 Thinking...

🤖 Answer:
According to the book, the mind affects our health and body by influencing the body's response to thoughts. The book states: "The body is the servant of the mind. It obeys the operations of the mind, whether they be deliberately chosen or automatically expressed." This means that the mind has the power to either build up or destroy the body, depending on the thoughts it chooses and encourages.

❓ Q3: What is the importance of having a definite purpose in life?
🔄 Thinking...

🤖 Answer:
A definite purpose in life is the centralizing point of one's thoughts, which should be steadily focuse

---
## 💬 CELL 11 — Interactive Chat Loop with Conversation History

Type your question → get an answer grounded in the book.  
Type **`quit`** or **`exit`** to stop. Type **`sources`** to see retrieved passages.

In [11]:
# ✅ FIX: ConversationBufferWindowMemory is deprecated in LangChain 1.x
#    We use a simple list to track conversation history instead.
from langchain_core.prompts import PromptTemplate

CHAT_PROMPT_TEMPLATE = """\
You are a wise assistant helping someone understand "As a Man Thinketh" by James Allen.
Answer using ONLY the provided book context. If the answer is not there, say so honestly.

PREVIOUS CONVERSATION:
{chat_history}

RELEVANT BOOK CONTEXT:
{context}

CURRENT QUESTION: {question}

ANSWER:"""

chat_prompt = PromptTemplate(
    template=CHAT_PROMPT_TEMPLATE,
    input_variables=["chat_history", "context", "question"]
)

# Simple conversation history (last 5 turns)
chat_history = []
MAX_HISTORY  = 5
last_sources = []

print("╔══════════════════════════════════════════════════════════╗")
print("║      📖 RAG CHATBOT — As a Man Thinketh                  ║")
print("║      Powered by Ollama (llama3.2) + LangChain 1.x        ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Ask anything about the book's teachings!                ║")
print("║  Commands: 'quit' → exit | 'sources' → show passages     ║")
print("╚══════════════════════════════════════════════════════════╝")
print()

while True:
    user_input = input("\n🧑 You: ").strip()

    if user_input.lower() in ["quit", "exit", "q"]:
        print("\n👋 Goodbye! Keep thinking great thoughts!")
        break

    if user_input.lower() == "sources":
        if last_sources:
            print("\n📚 Book passages used in last answer:")
            for i, src in enumerate(last_sources, 1):
                print(f"\n[Passage {i}]")
                print(src.page_content[:300] + "...")
        else:
            print("No sources yet — ask a question first.")
        continue

    if not user_input:
        continue

    # Step 1: Retrieve relevant chunks
    relevant_docs = retriever.invoke(user_input)
    last_sources  = relevant_docs
    context       = format_docs(relevant_docs)

    # Step 2: Build history string (last MAX_HISTORY turns)
    history_str = "\n".join(
        [f"Human: {h[0]}\nAssistant: {h[1]}" for h in chat_history[-MAX_HISTORY:]]
    )

    # Step 3: Fill prompt and call LLM
    filled_prompt = chat_prompt.format(
        chat_history=history_str,
        context=context,
        question=user_input
    )

    print("\n🤖 Assistant: ", end="", flush=True)
    response = llm.invoke(filled_prompt)
    print(response)

    # Step 4: Save to history
    chat_history.append((user_input, response))

    print("\n💡 Tip: Type 'sources' to see which passages were retrieved.")

╔══════════════════════════════════════════════════════════╗
║      📖 RAG CHATBOT — As a Man Thinketh                  ║
║      Powered by Ollama (llama3.2) + LangChain 1.x        ║
╠══════════════════════════════════════════════════════════╣
║  Ask anything about the book's teachings!                ║
║  Commands: 'quit' → exit | 'sources' → show passages     ║
╚══════════════════════════════════════════════════════════╝




🧑 You:  what is the book name



🤖 Assistant: The book name is "As a Man Thinketh" by James Allen.

💡 Tip: Type 'sources' to see which passages were retrieved.



🧑 You:  quit



👋 Goodbye! Keep thinking great thoughts!


---
## 💾 CELL 12 — BONUS: Reload Vector Store from Disk (Skip Re-embedding)
On your next session, run **this cell** instead of Cell 6 — loads existing ChromaDB instantly.

In [12]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

CHROMA_DIR = "../data/chroma_db"

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

# Load from disk — no re-embedding!
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model,
    collection_name="book_knowledge"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"✅ Loaded vector store from disk!")
print(f"📊 Chunks available: {vectorstore._collection.count()}")
print("🔍 Retriever ready!")

✅ Loaded vector store from disk!
📊 Chunks available: 84
🔍 Retriever ready!


---
## 📊 CELL 13 — BONUS: Visualize Similarity Scores
See exactly how relevant each retrieved chunk is to your question.

In [13]:
question = "How can a person change their life circumstances?"

docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
    question,
    k=5
)

print(f"❓ Question: {question}")
print(f"\n📊 Top 5 results with similarity scores:")
print("=" * 70)

for i, (doc, score) in enumerate(docs_with_scores, 1):
    bar_len = int(score * 30)
    bar = "█" * bar_len + "░" * (30 - bar_len)
    print(f"\n#{i} | Score: {score:.4f} |{bar}|")
    print(f"    {doc.page_content[:200]}...")

print()
print("💡 Score 1.0 = identical meaning | 0.0 = completely unrelated")

❓ Question: How can a person change their life circumstances?

📊 Top 5 results with similarity scores:

#1 | Score: 0.5293 |███████████████░░░░░░░░░░░░░░░|
    cannot fail to produce its results on the character and
circumstances. A man cannot _directly_ choose his circumstances, but
he can choose his thoughts, and so indirectly, yet surely, shape his
circum...

#2 | Score: 0.5042 |███████████████░░░░░░░░░░░░░░░|
    and discover itself through environment and circumstance, the outer
conditions of a person's life will always be found to be
harmoniously related to his inner state. This does not mean that a
man's ci...

#3 | Score: 0.4969 |██████████████░░░░░░░░░░░░░░░░|
    the result of a law which cannot err. This is just as true of those
who feel "out of harmony" with their surroundings as of those who
are contented with them.
As a progressive and evolving being, man ...

#4 | Score: 0.4966 |██████████████░░░░░░░░░░░░░░░░|
    result of a harmonious adjustment of the inner with the o

---
## 🎉 Complete! Here's What You Built:

| Step | Technology | Purpose |
|------|-----------|--------|
| Load | `langchain_community` TextLoader | Read book into Documents |
| Chunk | `langchain_text_splitters` | Split into 800-char pieces |
| Embed | `langchain_ollama` OllamaEmbeddings | Text → 768-dim vectors |
| Store | `langchain_chroma` Chroma | Persist vectors on disk |
| Retrieve | Similarity search | Find top-3 relevant chunks |
| Generate | `langchain_ollama` OllamaLLM | Answer using context |
| Chat | Manual history list | Multi-turn conversation |

### 🔬 Things to Experiment With:
- Change `chunk_size` (try 400, 600, 1200) — how do answers change?
- Change `k` in retriever (try 5) — more context = better or worse?
- Change `temperature` (try 0.0 or 0.5)
- Try another model: `ollama pull mistral` and change model name

📖 **Full learning guide:** `guides/RAG_Complete_Beginner_Guide.md`